In [1]:
!pip install opencv-contrib-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import cv2
import numpy as np
import sqlite3
import time
from datetime import datetime
from pathlib import Path

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
BASE_DIR = Path(r"C:\Python")
KNOWN_FACES_DIR = BASE_DIR / "known_faces"
MODEL_PATH = BASE_DIR / "face_model.yml"
DB_PATH = BASE_DIR / "attendance.db"

In [5]:
!curl -o haarcascade_frontalface_default.xml https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0           00:01              0
  5 908.3k   5  49152   0      0  17627      0   00:52   00:02   00:50  48567
 12 908.3k  12 112.0k   0      0  29914      0   00:31   00:03   00:28  55740
 15 908.3k  15 144.0k   0      0  29794      0   00:31   00:04   00:27  46474
 35 908.3k  35 320.0k   0      0  54957      0   00:16   00:05   00:11  78277
 48 908.3k  48 444.2k   0      0  65274      0   00:14   00:06   00:08  87602
 64 908.3k  64 588.2k   0      0  75354      0   00:12   00:07   00:05 103.7k
 97 908.3k  97 886.0k   0      0 98.41k      0   00:09   00:09         149.7k
100 908.3k 100 908.3k   0      0 99.86k      0   00:09   00:09         149.7k
100 908.3k 100 908.3k   0      0 99.86k      0   00:09   00:09         149.7k
100 908.3k 100 908.3k   0      0 99.85k      0   00:09   00:09 

In [15]:
KNOWN_FACES_DIR.mkdir(parents=True, exist_ok=True)

if not hasattr(cv2, "face"):
    raise ImportError(
        "cv2.face is not available. Run: pip install opencv-contrib-python"
    )

CASCADE_PATH = Path("haarcascade_frontalface_default.xml")

if not CASCADE_PATH.exists():
    raise FileNotFoundError(
        f"Haar Cascade not found: {CASCADE_PATH}"
    )

face_cascade = cv2.CascadeClassifier(str(CASCADE_PATH))

if face_cascade.empty():
    raise RuntimeError("Failed to load Haar Cascade.")

recognizer = cv2.face.LBPHFaceRecognizer_create()


def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL UNIQUE,
            face_id INTEGER NOT NULL UNIQUE,
            image_path TEXT,
            registered_at TEXT NOT NULL
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS attendance (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            name TEXT NOT NULL,
            date TEXT NOT NULL,
            time TEXT NOT NULL,
            timestamp TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    """)

    cur.execute("""
        CREATE UNIQUE INDEX IF NOT EXISTS
        idx_attendance_user_date
        ON attendance(user_id, date)
    """)

    conn.commit()
    conn.close()

    print("[OK] Database ready.")


def get_all_users():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("SELECT * FROM users ORDER BY id")

    rows = cur.fetchall()
    conn.close()

    return [dict(row) for row in rows]


def get_user_by_face_id(face_id):
    conn = get_connection()
    cur = conn.cursor()

    cur.execute(
        "SELECT * FROM users WHERE face_id = ?",
        (face_id,)
    )

    row = cur.fetchone()
    conn.close()

    return dict(row) if row else None


def get_next_face_id():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("SELECT MAX(face_id) FROM users")

    result = cur.fetchone()[0]
    conn.close()

    return 0 if result is None else result + 1


def add_user(name, face_id, image_path):
    conn = get_connection()
    cur = conn.cursor()

    try:
        cur.execute(
            """
            INSERT INTO users
            (name, face_id, image_path, registered_at)
            VALUES (?, ?, ?, ?)
            """,
            (
                name,
                face_id,
                image_path,
                datetime.now().isoformat()
            )
        )

        conn.commit()
        return True

    except sqlite3.IntegrityError:
        return False

    finally:
        conn.close()


def detect_faces(gray):
    return face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(80, 80)
    )


def capture_face_samples(name, num_samples=30):
    person_dir = KNOWN_FACES_DIR / name
    person_dir.mkdir(parents=True, exist_ok=True)

    old_images = list(person_dir.glob("*.jpg"))

    for image in old_images:
        try:
            image.unlink()
        except:
            pass

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("[ERROR] Cannot open webcam.")
        return False

    count = 0

    print()
    print("=" * 60)
    print(f"REGISTERING: {name}")
    print(f"TARGET SAMPLES: {num_samples}")
    print("Look directly at the camera.")
    print("Press Q inside the camera window to cancel.")
    print("=" * 60)

    while count < num_samples:

        ret, frame = cap.read()

        if not ret:
            print("[ERROR] Cannot read camera.")
            break

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        faces = detect_faces(gray)

        if len(faces) > 0:

            x, y, w, h = max(
                faces,
                key=lambda box: box[2] * box[3]
            )

            face_roi = gray[
                y:y + h,
                x:x + w
            ]

            if face_roi.size > 0:

                face_roi = cv2.resize(
                    face_roi,
                    (200, 200)
                )

                image_path = (
                    person_dir /
                    f"{count:03d}.jpg"
                )

                cv2.imwrite(
                    str(image_path),
                    face_roi
                )

                count += 1

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 255, 0),
                    2
                )

        cv2.putText(
            frame,
            f"Samples: {count}/{num_samples}",
            (10, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            "Press Q to cancel",
            (10, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 255),
            2
        )

        cv2.imshow(
            "Face Registration",
            frame
        )

        key = cv2.waitKey(50) & 0xFF

        if key == ord("q"):
            cap.release()
            cv2.destroyAllWindows()
            print("[INFO] Registration cancelled.")
            return False

    cap.release()
    cv2.destroyAllWindows()

    if count < 5:

        print(
            "[ERROR] Less than 5 face samples captured."
        )

        for image in person_dir.glob("*.jpg"):
            try:
                image.unlink()
            except:
                pass

        return False

    print(
        f"[OK] Captured {count} samples."
    )

    return True


def train_recognizer():

    users = get_all_users()

    if not users:
        print("[ERROR] No registered users.")
        return False

    faces = []
    labels = []

    for user in users:

        person_dir = (
            KNOWN_FACES_DIR /
            user["name"]
        )

        if not person_dir.exists():
            continue

        for image_path in sorted(
            person_dir.glob("*.jpg")
        ):

            image = cv2.imread(
                str(image_path),
                cv2.IMREAD_GRAYSCALE
            )

            if image is None:
                continue

            image = cv2.resize(
                image,
                (200, 200)
            )

            faces.append(image)
            labels.append(
                user["face_id"]
            )

    if not faces:
        print("[ERROR] No training images found.")
        return False

    recognizer.train(
        faces,
        np.array(
            labels,
            dtype=np.int32
        )
    )

    recognizer.write(
        str(MODEL_PATH)
    )

    print()
    print("=" * 60)
    print("MODEL TRAINING COMPLETED")
    print(f"Training images : {len(faces)}")
    print(f"Registered users: {len(set(labels))}")
    print(f"Model           : {MODEL_PATH}")
    print("=" * 60)

    return True


def load_recognizer():

    if not MODEL_PATH.exists():
        return False

    try:
        recognizer.read(
            str(MODEL_PATH)
        )
        return True

    except Exception as e:
        print(
            "[ERROR] Model loading failed:",
            e
        )
        return False


def recognize_face(
    face_img,
    threshold=70
):

    if face_img is None:
        return None, None

    if face_img.size == 0:
        return None, None

    face_img = cv2.resize(
        face_img,
        (200, 200)
    )

    try:
        label, confidence = recognizer.predict(
            face_img
        )

    except Exception:
        return None, None

    if confidence < threshold:
        return (
            int(label),
            float(confidence)
        )

    return (
        None,
        float(confidence)
    )


def register_user(
    name,
    num_samples=30
):

    name = str(name).strip()

    if not name:
        print("[ERROR] Name cannot be empty.")
        return

    users = get_all_users()

    existing_names = [
        user["name"].lower()
        for user in users
    ]

    if name.lower() in existing_names:
        print(
            f"[ERROR] '{name}' is already registered."
        )
        return

    success = capture_face_samples(
        name,
        num_samples
    )

    if not success:
        return

    face_id = get_next_face_id()

    folder = KNOWN_FACES_DIR / name

    if not add_user(
        name,
        face_id,
        str(folder)
    ):
        print("[ERROR] Could not add user.")
        return

    print(
        f"[OK] User '{name}' added with Face ID {face_id}."
    )

    if train_recognizer():

        print()
        print("=" * 60)
        print("REGISTRATION SUCCESSFUL")
        print(f"Name   : {name}")
        print(f"Face ID: {face_id}")
        print("=" * 60)

    else:

        print(
            "[ERROR] User added but model training failed."
        )


def mark_attendance(
    user_id,
    name
):

    conn = get_connection()
    cur = conn.cursor()

    now = datetime.now()

    today = now.strftime(
        "%Y-%m-%d"
    )

    current_time = now.strftime(
        "%H:%M:%S"
    )

    timestamp = now.isoformat()

    cur.execute(
        """
        SELECT id
        FROM attendance
        WHERE user_id = ?
        AND date = ?
        """,
        (
            user_id,
            today
        )
    )

    if cur.fetchone():

        conn.close()

        return False

    cur.execute(
        """
        INSERT INTO attendance
        (
            user_id,
            name,
            date,
            time,
            timestamp
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            user_id,
            name,
            today,
            current_time,
            timestamp
        )
    )

    conn.commit()
    conn.close()

    return True


def run_attendance(
    threshold=70
):

    if not load_recognizer():

        print(
            "[ERROR] No trained model found."
        )

        print(
            "Run register_user('Your Name') first."
        )

        return

    cap = cv2.VideoCapture(0)

    if not cap.isOpened():

        print(
            "[ERROR] Cannot open webcam."
        )

        return

    print()
    print("=" * 60)
    print("ATTENDANCE MODE")
    print("Press Q inside the camera window to stop.")
    print("=" * 60)

    last_seen = {}

    cooldown = 10

    while True:

        ret, frame = cap.read()

        if not ret:
            print(
                "[ERROR] Camera frame error."
            )
            break

        gray = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2GRAY
        )

        faces = detect_faces(gray)

        status = "Looking for face..."
        status_color = (
            200,
            200,
            200
        )

        for x, y, w, h in faces:

            face_roi = gray[
                y:y + h,
                x:x + w
            ]

            face_id, confidence = recognize_face(
                face_roi,
                threshold
            )

            if face_id is not None:

                user = get_user_by_face_id(
                    face_id
                )

                if user is not None:

                    name = user["name"]

                    current_time = time.time()

                    if (
                        current_time
                        - last_seen.get(
                            name,
                            0
                        )
                        > cooldown
                    ):

                        marked = mark_attendance(
                            user["id"],
                            name
                        )

                        last_seen[name] = (
                            current_time
                        )

                        if marked:

                            status = (
                                f"ATTENDANCE MARKED: "
                                f"{name}"
                            )

                            status_color = (
                                0,
                                255,
                                0
                            )

                            print(
                                f"[OK] Attendance marked: "
                                f"{name}"
                            )

                        else:

                            status = (
                                f"Already marked today: "
                                f"{name}"
                            )

                            status_color = (
                                0,
                                200,
                                255
                            )

                    else:

                        status = (
                            f"Recognized: {name}"
                        )

                        status_color = (
                            0,
                            255,
                            255
                        )

                    cv2.rectangle(
                        frame,
                        (x, y),
                        (x + w, y + h),
                        (0, 255, 0),
                        2
                    )

                    cv2.putText(
                        frame,
                        f"{name} "
                        f"({confidence:.0f})",
                        (
                            x,
                            max(y - 10, 20)
                        ),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 255, 0),
                        2
                    )

                else:

                    status = (
                        "REJECTED - Unknown"
                    )

                    status_color = (
                        0,
                        0,
                        255
                    )

                    cv2.rectangle(
                        frame,
                        (x, y),
                        (x + w, y + h),
                        (0, 0, 255),
                        2
                    )

                    cv2.putText(
                        frame,
                        "REJECTED",
                        (
                            x,
                            max(y - 10, 20)
                        ),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,
                        (0, 0, 255),
                        2
                    )

            else:

                status = (
                    "REJECTED - Not recognized"
                )

                status_color = (
                    0,
                    0,
                    255
                )

                cv2.rectangle(
                    frame,
                    (x, y),
                    (x + w, y + h),
                    (0, 0, 255),
                    2
                )

                confidence_text = (
                    f"{confidence:.0f}"
                    if confidence is not None
                    else "?"
                )

                cv2.putText(
                    frame,
                    f"REJECTED "
                    f"({confidence_text})",
                    (
                        x,
                        max(y - 10, 20)
                    ),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 0, 255),
                    2
                )

        cv2.rectangle(
            frame,
            (0, 0),
            (
                frame.shape[1],
                45
            ),
            (30, 30, 30),
            -1
        )

        cv2.putText(
            frame,
            status,
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            status_color,
            2
        )

        current_datetime = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        cv2.putText(
            frame,
            current_datetime,
            (
                max(
                    frame.shape[1] - 210,
                    10
                ),
                30
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (180, 180, 180),
            1
        )

        cv2.imshow(
            "Face Recognition Attendance",
            frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    print("[INFO] Attendance mode closed.")


def show_today_attendance():

    conn = get_connection()
    cur = conn.cursor()

    today = datetime.now().strftime(
        "%Y-%m-%d"
    )

    cur.execute(
        """
        SELECT *
        FROM attendance
        WHERE date = ?
        ORDER BY time
        """,
        (today,)
    )

    records = cur.fetchall()

    conn.close()

    print()
    print("=" * 60)
    print(
        f"ATTENDANCE - {today}"
    )
    print("=" * 60)

    if not records:

        print(
            "No attendance recorded today."
        )

    else:

        print(
            f"{'ID':<8}"
            f"{'Name':<25}"
            f"{'Time':<15}"
        )

        print("-" * 60)

        for record in records:

            print(
                f"{record['user_id']:<8}"
                f"{record['name']:<25}"
                f"{record['time']:<15}"
            )

    print("=" * 60)


def show_users():

    users = get_all_users()

    print()
    print("=" * 60)
    print("REGISTERED USERS")
    print("=" * 60)

    if not users:

        print(
            "No users registered."
        )

    else:

        for user in users:

            print(
                f"Face ID: "
                f"{user['face_id']:<5} "
                f"Name: "
                f"{user['name']}"
            )

    print("=" * 60)


init_db()

print()
print("=" * 60)
print("FACE RECOGNITION ATTENDANCE SYSTEM")
print("=" * 60)
print("Code loaded successfully.")
print()
print("Register:")
print("register_user('Raghavi')")
print()
print("Start attendance:")
print("run_attendance()")
print()
print("Today's attendance:")
print("show_today_attendance()")
print()
print("Registered users:")
print("show_users()")
print("=" * 60)

[OK] Database ready.

FACE RECOGNITION ATTENDANCE SYSTEM
Code loaded successfully.

Register:
register_user('Raghavi')

Start attendance:
run_attendance()

Today's attendance:
show_today_attendance()

Registered users:
show_users()


In [16]:
register_user("Raghavi")

[ERROR] 'Raghavi' is already registered.


In [17]:
run_attendance()


ATTENDANCE MODE
Press Q inside the camera window to stop.
[OK] Attendance marked: Raghavi
[INFO] Attendance mode closed.


In [18]:
show_today_attendance()


ATTENDANCE - 2026-09-27
ID      Name                     Time           
------------------------------------------------------------
4       Raghavi                  17:02:52       
